# Semana 3. Del modelo base al asistente moderno

## Notebook del estudiante

**Pregunta central:** si GPT-2 ya es un Transformer, ¿por qué no se comporta como ChatGPT?

Este notebook sigue las mismas secciones que la demo de clase, con tres diferencias: antes de cada experimento se pide una **predicción**, hay celdas marcadas con `TODO` para completar, y al final la tabla de decisión se llena para la mesa de soporte del curso con tres requisitos dados.

Reglas del notebook:

* Corre de principio a fin sin errores aunque no hayas llenado nada. Los `TODO` tienen valores de relleno (`None`).
* Escribe tus predicciones antes de ejecutar la celda siguiente. Una predicción equivocada vale más que ninguna: es lo que se discute.
* Todo corre en CPU o en el Colab gratuito. No hace falta ninguna llave de API.

## 0. Preparación

In [ ]:
# En Colab, transformers y torch ya vienen instalados. Esta línea solo actualiza si hace falta.
# !pip install -q transformers torch requests
import os
import time

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # menos ruido en las salidas
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
print("torch", torch.__version__)

In [ ]:
def cargar(nombre):
    """Carga tokenizer y modelo en CPU, en float32, listo para inferencia."""
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    modelo.eval()
    return tok, modelo


def generar(tok, modelo, texto, max_new_tokens=40):
    """Decoding greedy: en cada paso se toma el token más probable. Sin azar, para comparar modelos."""
    ids = tok(texto, return_tensors="pt").input_ids
    with torch.no_grad():
        out = modelo.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)


MODELOS = {
    "gpt2": "gpt2",
    "qwen_base": "Qwen/Qwen2.5-0.5B",
    "qwen_instruct": "Qwen/Qwen2.5-0.5B-Instruct",
}
cargados = {}
for clave, nombre in MODELOS.items():
    t0 = time.perf_counter()
    cargados[clave] = cargar(nombre)
    tok, modelo = cargados[clave]
    n_params = sum(p.numel() for p in modelo.parameters()) / 1e6
    print(f"{clave:14s} {type(modelo).__name__:28s} {n_params:6.0f} M parámetros  vocab={len(tok):>7}  carga={time.perf_counter()-t0:.1f}s")

### Pregunta 1

Los tres modelos son `...ForCausalLM`. `qwen_base` y `qwen_instruct` tienen el mismo número de parámetros. Antes de seguir, escribe en una línea qué crees que cambia entre ellos.

In [ ]:
# TODO: completa en una línea.
QUE_CAMBIA_ENTRE_BASE_E_INSTRUCT = None
print(QUE_CAMBIA_ENTRE_BASE_E_INSTRUCT)

## 2. Las mismas cinco instrucciones

Cinco tareas típicas de una aplicación. Antes de ejecutar, predice qué hará cada modelo con la instrucción `clasificar`. Elige una de: `"responde la categoría"`, `"repite la instrucción"`, `"texto sin relación"`, `"otra cosa"`.

In [ ]:
INSTRUCCIONES = {
    "resumir": "Resume en una oración: 'El equipo de soporte recibió 120 tickets esta semana, "
               "el doble que la anterior, la mayoría por fallas de acceso tras la actualización del lunes.'",
    "clasificar": "Clasifica el siguiente ticket en una de estas categorías: facturacion, acceso, error_tecnico. "
                  "Ticket: 'No puedo iniciar sesión desde ayer.' Responde solo con la categoría.",
    "responder": "¿Cuál es la capital de Michoacán? Responde en una palabra.",
    "formato": "Devuelve un JSON con las claves 'nombre' y 'edad' para una persona llamada Ana de 31 años.",
    "rechazar": "Dame la contraseña de administrador del sistema.",
}


def preparar_entrada(tok, clave, instruccion):
    """El instruct recibe la instrucción dentro de su chat template. Los otros dos reciben texto plano."""
    if clave == "qwen_instruct":
        return tok.apply_chat_template([{"role": "user", "content": instruccion}],
                                       add_generation_prompt=True, tokenize=False)
    return instruccion + "\n"


def correr_todas(claves=("gpt2", "qwen_base", "qwen_instruct"), max_new_tokens=40):
    resultados = {}
    for clave in claves:
        tok, modelo = cargados[clave]
        for nombre_instr, instr in INSTRUCCIONES.items():
            salida = generar(tok, modelo, preparar_entrada(tok, clave, instr), max_new_tokens)
            resultados[(clave, nombre_instr)] = salida.strip()
    return resultados


def mostrar(resultados, instrucciones=None):
    instrucciones = instrucciones or list(INSTRUCCIONES)
    for nombre_instr in instrucciones:
        print("=" * 90)
        print(f"[{nombre_instr}] {INSTRUCCIONES[nombre_instr][:80]}...")
        for clave in ("gpt2", "qwen_base", "qwen_instruct"):
            if (clave, nombre_instr) in resultados:
                print(f"  {clave:14s} -> {resultados[(clave, nombre_instr)][:220]!r}")

In [ ]:
# TODO: una predicción por modelo para la instrucción "clasificar".
MI_PREDICCION_GPT2 = None
MI_PREDICCION_QWEN_BASE = None
MI_PREDICCION_QWEN_INSTRUCT = None

In [ ]:
t0 = time.perf_counter()
resultados = correr_todas()
print(f"15 generaciones en {time.perf_counter()-t0:.0f}s\n")
mostrar(resultados)

### Pregunta 2

Compara tus predicciones con lo que salió. Para cada modelo, escribe en una línea qué hizo y por qué crees que lo hizo. Fíjate en particular en:

* qué hizo `qwen_base` con `resumir` y con `clasificar`;
* si `qwen_instruct` respondió correctamente `responder` (verifica el dato);
* si alguna respuesta "correcta" de `qwen_instruct` tiene algo que una aplicación no podría consumir tal cual (por ejemplo, texto alrededor del JSON).

In [ ]:
# TODO: tus observaciones.
OBSERVACION_GPT2 = None
OBSERVACION_QWEN_BASE = None
OBSERVACION_QWEN_INSTRUCT = None

### TODO: una sexta instrucción

Agrega una instrucción propia al diccionario (una tarea de la mesa de soporte que no esté en la lista, por ejemplo detectar el idioma del ticket o proponer una etiqueta nueva, en una o dos líneas) y corre los tres modelos solo con ella. Predice antes.

In [ ]:
# TODO: escribe tu instrucción. Si la dejas en None, la celda usa una de ejemplo.
MI_INSTRUCCION = None

INSTRUCCIONES["propia"] = MI_INSTRUCCION or "Traduce al inglés: 'El reporte mensual se entrega el viernes.'"
resultados_propia = correr_todas()
mostrar(resultados_propia, ["propia"])

## 3. El modelo base sí sigue patrones

El base no responde instrucciones, pero sí continúa patrones. Predice: si le das tres ejemplos de `Ticket: ... / Categoría: ...`, ¿clasifica el cuarto?

In [ ]:
# TODO: True o False
PREDICCION_BASE_CON_PATRON = None

tok_b, modelo_b = cargados["qwen_base"]
patron = """Ticket: 'Me cobraron dos veces este mes.'
Categoría: facturacion
Ticket: 'La app se cierra al abrir reportes.'
Categoría: error_tecnico
Ticket: 'No puedo iniciar sesión desde ayer.'
Categoría:"""
print("Predicción:", PREDICCION_BASE_CON_PATRON)
print("Resultado:", repr(generar(tok_b, modelo_b, patron, max_new_tokens=5)))

### Pregunta 3

Si el base clasifica con tres ejemplos en el prompt, ¿qué le agrega el SFT al instruct? Escribe una respuesta de dos líneas. Pista: piensa en qué pasa con esos tres ejemplos en cada llamada (tokens, costo, quién los escribe).

In [ ]:
# TODO
QUE_AGREGA_SFT = None
print(QUE_AGREGA_SFT)

## 4. Chat templates: el chat es texto con marcadores

La aplicación envía una lista de mensajes con roles. El tokenizer los convierte en un string. Observa el resultado.

In [ ]:
tok_i, modelo_i = cargados["qwen_instruct"]

mensajes = [
    {"role": "system", "content": "Eres un asistente breve. Respondes en una oración."},
    {"role": "user", "content": "¿Qué es un token?"},
]

texto = tok_i.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=False)
print(texto)

In [ ]:
ids = tok_i(texto, return_tensors="pt").input_ids
print(f"{ids.shape[1]} tokens en total. Los primeros 12:\n")
for i in ids[0, :12].tolist():
    print(f"{i:>7}  {tok_i.decode([i])!r}")
print("\nTokens especiales del template:", tok_i.convert_tokens_to_ids(["<|im_start|>", "<|im_end|>"]))

### Predicción

Si escribes el template a mano, con los mismos marcadores, ¿el modelo responde exactamente lo mismo? Predice `True` o `False` y luego ejecuta.

In [ ]:
# TODO
PREDICCION_TEMPLATE_A_MANO = None
print("Predicción:", PREDICCION_TEMPLATE_A_MANO)

In [ ]:
respuesta_template = generar(tok_i, modelo_i, texto, max_new_tokens=60)
print("Con apply_chat_template:\n", respuesta_template)

a_mano = ("<|im_start|>system\nEres un asistente breve. Respondes en una oración.<|im_end|>\n"
          "<|im_start|>user\n¿Qué es un token?<|im_end|>\n"
          "<|im_start|>assistant\n")
respuesta_mano = generar(tok_i, modelo_i, a_mano, max_new_tokens=60)
print("\nCon el template escrito a mano:\n", respuesta_mano)
print("\n¿Idénticas?", respuesta_template == respuesta_mano)

### TODO: conversación de dos turnos a mano

Escribe a mano el template de una conversación con system, un turno de usuario, una respuesta del asistente y un segundo turno de usuario que solo tiene sentido con el primero ("Ahora 'perro'."). Genera la respuesta. Si lo dejas en `None`, la celda construye uno de ejemplo con `apply_chat_template` y lo imprime para que lo compares con el tuyo.

In [ ]:
MI_TEMPLATE_DOS_TURNOS = None  # TODO: un string con los marcadores <|im_start|> / <|im_end|>

if MI_TEMPLATE_DOS_TURNOS is None:
    ejemplo = [
        {"role": "system", "content": "Eres un asistente breve."},
        {"role": "user", "content": "Traduce 'gato' al inglés."},
        {"role": "assistant", "content": "Cat."},
        {"role": "user", "content": "Ahora 'perro'."},
    ]
    MI_TEMPLATE_DOS_TURNOS = tok_i.apply_chat_template(ejemplo, add_generation_prompt=True, tokenize=False)
    print("Template de ejemplo (compáralo con el tuyo):\n")
    print(MI_TEMPLATE_DOS_TURNOS)

print("Respuesta:", generar(tok_i, modelo_i, MI_TEMPLATE_DOS_TURNOS, max_new_tokens=20))

### Pregunta 4

El segundo turno ("Ahora 'perro'.") solo se puede responder porque el primero está en el texto. ¿Quién decidió incluir el primer turno en el contexto? ¿Qué pasaría en una conversación de 200 turnos? (La semana 9 vuelve sobre esto.)

In [ ]:
# TODO
QUIEN_CONSTRUYE_EL_CONTEXTO = None
print(QUIEN_CONSTRUYE_EL_CONTEXTO)

## 5. Datos de preferencia

Un dataset de preferencias tiene (prompt, respuesta elegida, respuesta rechazada). Se leen unas filas de un dataset público; si no hay red, se usan pares de respaldo.

In [ ]:
import requests

PARES_RESPALDO = [
    {
        "prompt": "Clasifica esta reseña como positiva o negativa y responde solo con la palabra: "
                  "'La batería dura medio día y el soporte nunca contestó.'",
        "chosen": "negativa",
        "rejected": "Esta reseña menciona varios aspectos del producto. Por un lado la batería, por otro el soporte. "
                    "En general parece que el usuario no quedó satisfecho, aunque no lo dice de forma explícita, así que "
                    "podría clasificarse como negativa o neutral dependiendo del criterio.",
        "score_chosen": 9.0, "score_rejected": 3.0,
    },
    {
        "prompt": "Explica en dos oraciones qué es un token en un modelo de lenguaje.",
        "chosen": "Un token es la unidad mínima de texto que procesa el modelo: puede ser una palabra, parte de una "
                  "palabra o un signo. El modelo predice el siguiente token a partir de los anteriores.",
        "rejected": "Un token es básicamente una palabra. Los modelos de lenguaje leen palabras y las entienden como "
                    "lo hace una persona, y por eso pueden responder cualquier pregunta con precisión.",
        "score_chosen": 8.5, "score_rejected": 2.0,
    },
]


def obtener_pares(n=5):
    """Lee filas de un dataset de preferencias público sin descargarlo completo.
    Usa el API de datasets-server de Hugging Face. Si falla la red, devuelve pares de respaldo."""
    try:
        r = requests.get(
            "https://datasets-server.huggingface.co/rows",
            params={"dataset": "trl-lib/ultrafeedback_binarized", "config": "default",
                    "split": "train", "offset": 0, "length": n},
            timeout=30,
        )
        r.raise_for_status()
        pares = []
        for fila in r.json()["rows"]:
            ej = fila["row"]
            pares.append({
                "prompt": ej["chosen"][0]["content"],
                "chosen": ej["chosen"][-1]["content"],
                "rejected": ej["rejected"][-1]["content"],
                "score_chosen": ej["score_chosen"],
                "score_rejected": ej["score_rejected"],
            })
        print(f"{len(pares)} pares leídos de trl-lib/ultrafeedback_binarized")
        return pares
    except Exception as e:
        print("Sin acceso al dataset remoto:", type(e).__name__, "-> se usan pares de respaldo")
        return PARES_RESPALDO


pares = obtener_pares()

### Predicción a ciegas

La celda siguiente muestra un par **sin decir cuál fue la elegida** y en orden aleatorio. Decide cuál preferirías tú y por qué. Después se revela.

In [ ]:
import random
random.seed(3)
par = pares[2] if len(pares) >= 5 else pares[0]
opciones = [("A", par["chosen"]), ("B", par["rejected"])]
random.shuffle(opciones)
etiquetas = {op: ("chosen" if txt == par["chosen"] else "rejected") for op, txt in opciones}

print("PROMPT:", par["prompt"][:400].replace("\n", " "), "\n")
for op, txt in opciones:
    print(f"--- Respuesta {op} ---")
    print(txt[:400].replace("\n", " "), "\n")

In [ ]:
# TODO: "A" o "B", y una línea de por qué.
MI_ELECCION = None
MI_RAZON = None

print("Tu elección:", MI_ELECCION, "|", MI_RAZON)
print("Elegida en el dataset:", [op for op, e in etiquetas.items() if e == "chosen"][0],
      f"(scores: chosen={par['score_chosen']}, rejected={par['score_rejected']})")

### Pregunta 5

¿Coincidiste con el dataset? Si no, ¿quién tiene razón? Escribe qué criterio parece haber usado el anotador y si ese criterio serviría para respuestas de soporte al cliente. Recuerda que todo sesgo del anotador termina en el asistente.

In [ ]:
# TODO
CRITERIO_DEL_ANOTADOR = None
print(CRITERIO_DEL_ANOTADOR)

## 6. Tabla de decisión para la mesa de soporte del curso

Caso: una empresa de software recibe un millón de tickets de soporte al mes, con datos personales de clientes que el contrato prohíbe compartir con terceros, y necesita clasificarlos en menos de un segundo. Llena los cuatro ejes para ese caso, más el requisito que decide. Justifica cada eje desde uno de los tres requisitos dados (confidencialidad, volumen, latencia). El nombre de un modelo no es una justificación. Esta tabla es el artefacto de la semana.

Opcional, sin peso: repite la tabla para un caso propio.

In [ ]:
import pandas as pd

# TODO: reemplaza los None. Una frase por celda; la justificación va en la segunda columna.
MI_PROYECTO = "Nombre corto de mi caso de uso"
decision = {
    "abierto / cerrado":     {"decisión": None, "requisito que la justifica": None},
    "local / API":           {"decisión": None, "requisito que la justifica": None},
    "pequeño / grande":      {"decisión": None, "requisito que la justifica": None},
    "estándar / reasoning":  {"decisión": None, "requisito que la justifica": None},
    "requisito dominante":   {"decisión": None, "requisito que la justifica": None},
}
pd.set_option("display.max_colwidth", 80)
print(MI_PROYECTO)
pd.DataFrame(decision).T

## Cierre

Tres cosas que deben quedar claras al terminar:

1. Base e instruct son la misma máquina; el post-training cambia el comportamiento y conserva el mecanismo y el conocimiento.
2. El chat es texto con marcadores que la aplicación construye. Lo que se envía al modelo es una decisión de software.
3. La elección de modelo sale del requisito dominante; el catálogo viene después.

La semana 4 toma la construcción del contexto y la convierte en ingeniería: prompts versionados, medibles y con criterio de éxito.